In [1]:
import pandas as pd
import numpy as np

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [2]:
balls = pd.read_parquet(
    "../ml-service/data/processed/v3_gamma/clean_deliveries.parquet"
)
matches = pd.read_parquet("../ml-service/data/processed/v3_gamma/clean_matches.parquet")

NameError: name 'pd' is not defined

In [3]:
balls.head()

,matchId,inning,over,ball,batting_team,bowling_team,batsman,non_striker,bowler,batsman_runs,isWide,isNoBall,Byes,LegByes,Penalty,player_dismissed,date,total_runs
0,335982,0,0,1,Kolkata Knight Riders,Royal Challengers Bangalore,SC Ganguly,BB McCullum,P Kumar,0,0,0,0.0,1.0,0.0,Not Out,2008-04-18,1.0
1,335982,0,0,2,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,SC Ganguly,P Kumar,0,0,0,0.0,0.0,0.0,Not Out,2008-04-18,0.0
2,335982,0,0,3,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,SC Ganguly,P Kumar,0,1,0,0.0,0.0,0.0,Not Out,2008-04-18,1.0
3,335982,0,0,4,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,SC Ganguly,P Kumar,0,0,0,0.0,0.0,0.0,Not Out,2008-04-18,0.0
4,335982,0,0,5,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,SC Ganguly,P Kumar,0,0,0,0.0,0.0,0.0,Not Out,2008-04-18,0.0


In [4]:
balls.columns

Index(['matchId', 'inning', 'over', 'ball', 'batting_team', 'bowling_team',
       'batsman', 'non_striker', 'bowler', 'batsman_runs', 'isWide',
       'isNoBall', 'Byes', 'LegByes', 'Penalty', 'player_dismissed', 'date',
       'total_runs'],
      dtype='object')

In [5]:
balls.groupby(["matchId", "inning", "over", "ball"]).size().value_counts()

1    273788
2        31
Name: count, dtype: int64

In [4]:
duplicates = balls[
    balls.duplicated(["matchId", "inning", "over", "ball"], keep=False)
].sort_values(["matchId", "inning", "over", "ball"])
duplicates.head()

,matchId,inning,over,ball,batting_team,bowling_team,batsman,non_striker,bowler,batsman_runs,isWide,isNoBall,Byes,LegByes,Penalty,player_dismissed,date,total_runs
5635,336005,1,6,1,Rajasthan Royals,Chennai Super Kings,GC Smith,SA Asnodkar,JA Morkel,0,0,0,0.0,0.0,0.0,Not Out,2008-05-04,0.0
5636,336005,1,6,1,Rajasthan Royals,Chennai Super Kings,GC Smith,SA Asnodkar,JA Morkel,0,0,0,0.0,4.0,0.0,Not Out,2008-05-04,4.0
9216,336024,0,18,1,Mumbai Indians,Deccan Chargers,PR Shah,YV Takawale,DP Vijaykumar,0,1,0,0.0,0.0,0.0,Not Out,2008-05-18,1.0
9217,336024,0,18,1,Mumbai Indians,Deccan Chargers,PR Shah,YV Takawale,DP Vijaykumar,1,0,0,0.0,0.0,0.0,Not Out,2008-05-18,1.0
35415,419141,1,1,1,Deccan Chargers,Rajasthan Royals,AC Gilchrist,VVS Laxman,SR Watson,0,1,0,0.0,0.0,0.0,Not Out,2010-04-05,1.0


In [5]:
len(duplicates)

62

In [6]:
balls = balls.groupby(["matchId", "inning", "over", "ball"], as_index=False).agg(
    {
        "batsman_runs": "sum",
        "isWide": "sum",
        "isNoBall": "sum",
        "Byes": "sum",
        "LegByes": "sum",
        "Penalty": "sum",
        "total_runs": "sum",  # important
        "batting_team": "first",
        "bowling_team": "first",
        "batsman": "first",
        "non_striker": "first",
        "bowler": "first",
        "player_dismissed": lambda x: (
            x[x != "Not Out"].iloc[0] if any(x != "Not Out") else "Not Out"
        ),
        "date": "first",
    }
)

In [7]:
balls.groupby(["matchId", "inning", "over", "ball"]).size().value_counts()

1    273819
Name: count, dtype: int64

In [8]:
balls = balls.sort_values(["matchId", "inning", "over", "ball"]).reset_index(drop=True)

In [9]:
balls.groupby(["matchId", "inning"])["matchId"].nunique().value_counts()

matchId
1    2284
Name: count, dtype: int64

In [10]:
mask = balls["isNoBall"] > 1

balls.loc[mask, "batsman_runs"] += balls.loc[mask, "isNoBall"] - 1
balls.loc[mask, "isNoBall"] = 1

In [11]:
balls["repeat"] = np.where(balls["isWide"] > 0, balls["isWide"], 1)

balls = balls.loc[balls.index.repeat(balls["repeat"])].copy()

balls.loc[balls["isWide"] > 0, "isWide"] = 1

balls.drop(columns=["repeat"], inplace=True)

In [12]:
balls["is_legal"] = ((balls["isWide"] == 0) & (balls["isNoBall"] == 0)).astype(int)

balls["ball"] = balls.groupby(["matchId", "inning", "over"])["is_legal"].cumsum()

balls["ball"] = balls["ball"].replace(0, np.nan)
balls["ball"] = balls.groupby(["matchId", "inning", "over"])["ball"].ffill().fillna(1)

In [13]:
balls = balls.reset_index(drop=True)

In [14]:
balls = balls[balls["ball"] <= 6]

In [15]:
balls = balls.rename(columns={"ball": "legal_ball"})

In [16]:
balls["legal_ball"].unique()

array([1., 2., 3., 4., 5., 6.])

In [17]:
balls.groupby(["matchId", "inning", "over"])["is_legal"].sum().max()

np.int64(6)

In [18]:
balls["legal_ball_1"] = (balls["isWide"] == 0) & (balls["isNoBall"] == 0)

balls["balls_bowled"] = balls.groupby(["matchId", "inning"])["legal_ball_1"].cumsum()

balls["balls_remaining"] = 120 - balls["balls_bowled"]

balls.drop(columns=["balls_bowled"], inplace=True)

In [19]:
balls.groupby(["matchId", "inning"])["legal_ball_1"].sum().max()

np.int64(120)

In [20]:
balls = balls.reset_index(drop=True)

In [21]:
balls["balls_remaining"].unique()

array([119, 118, 117, 116, 115, 114, 113, 112, 111, 110, 109, 108, 107,
       106, 105, 104, 103, 102, 101, 100,  99,  98,  97,  96,  95,  94,
        93,  92,  91,  90,  89,  88,  87,  86,  85,  84,  83,  82,  81,
        80,  79,  78,  77,  76,  75,  74,  73,  72,  71,  70,  69,  68,
        67,  66,  65,  64,  63,  62,  61,  60,  59,  58,  57,  56,  55,
        54,  53,  52,  51,  50,  49,  48,  47,  46,  45,  44,  43,  42,
        41,  40,  39,  38,  37,  36,  35,  34,  33,  32,  31,  30,  29,
        28,  27,  26,  25,  24,  23,  22,  21,  20,  19,  18,  17,  16,
        15,  14,  13,  12,  11,  10,   9,   8,   7,   6,   5,   4,   3,
         2,   1,   0, 120])

In [22]:
balls["over_number"] = balls["over"].astype(int) + 1
balls["phase"] = np.select(
    [balls["over_number"] <= 6, balls["over_number"] <= 15], [0, 1], default=2
)
# balls.drop(columns=['over_number'], inplace=True)

In [23]:
balls["total_runs"] = (
    balls["batsman_runs"]
    + balls["isWide"]
    + balls["isNoBall"]
    + balls["Byes"]
    + balls["LegByes"]
    + balls["Penalty"]
)
balls["current_score"] = balls.groupby(["matchId", "inning"])["total_runs"].cumsum()
# balls.drop(columns=['total_runs'], inplace=True)

In [24]:
balls[["over", "legal_ball", "current_score", "isWide", "isNoBall"]].head(10)

,over,legal_ball,current_score,isWide,isNoBall
0,0,1.0,1.0,0,0
1,0,2.0,1.0,0,0
2,0,2.0,2.0,1,0
3,0,3.0,2.0,0,0
4,0,4.0,2.0,0,0
5,0,5.0,2.0,0,0
6,0,6.0,3.0,0,0
7,1,1.0,3.0,0,0
8,1,2.0,7.0,0,0
9,1,3.0,11.0,0,0


In [25]:
balls["current_score"].unique().max()

np.float64(287.0)

In [26]:
balls.loc[balls["Penalty"] == 5, "batsman_runs"] = 5

In [27]:
balls["batsman_runs"] = balls["batsman_runs"] + balls["Byes"] + balls["LegByes"]
balls["batsman_runs"].unique()

array([1., 0., 4., 6., 2., 5., 3., 7.])

In [28]:
balls.loc[balls["batsman_runs"] > 6, "batsman_runs"] = 6

In [29]:
balls = balls.reset_index(drop=True)

In [30]:
len(balls["player_dismissed"].unique())

656

In [31]:
balls["is_wicket"] = (balls["player_dismissed"] != "Not Out").astype(int)
balls["wickets_fallen"] = balls.groupby(["matchId", "inning"])["is_wicket"].cumsum()

In [32]:
balls["wickets_fallen"].unique()

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10])

In [33]:
balls = balls.reset_index(drop=True)

In [34]:
first_innings_score = (
    balls[balls["inning"] == 0].groupby("matchId")["current_score"].max()
)
balls["target"] = balls["matchId"].map(first_innings_score)
balls.loc[balls["inning"] == 1, "target"] = balls["target"] + 1
balls.loc[balls["inning"] == 0, "target"] = 0

In [35]:
balls[balls["target"] > 0].head(2)

,matchId,inning,over,legal_ball,batsman_runs,isWide,isNoBall,Byes,LegByes,Penalty,...,date,is_legal,legal_ball_1,balls_remaining,over_number,phase,current_score,is_wicket,wickets_fallen,target
129,335982,1,0,1.0,1.0,0,0,0.0,0.0,0.0,...,2008-04-18,1,True,119,1,0,1.0,0,0,223.0
130,335982,1,0,1.0,0.0,1,0,0.0,0.0,0.0,...,2008-04-18,0,False,119,1,0,2.0,0,0,223.0


In [36]:
balls["target"].unique().max()

np.float64(288.0)

In [37]:
over_runs = (
    balls.groupby(["matchId", "inning", "over_number"])["total_runs"]
    .sum()
    .reset_index(name="over_runs")
)
over_runs["last_over_runs"] = over_runs.groupby(["matchId", "inning"])[
    "over_runs"
].shift(1)
balls = balls.merge(
    over_runs[["matchId", "inning", "over_number", "last_over_runs"]],
    on=["matchId", "inning", "over_number"],
    how="left",
)
balls["last_over_runs"] = balls["last_over_runs"].fillna(0)
balls["last_over_runs"] = balls["last_over_runs"].astype(int)

In [38]:
balls["total_balls"] = balls.groupby(["matchId", "inning", "over"]).cumcount() + 1

In [39]:
balls.groupby(
    ["matchId", "inning", "over", "legal_ball", "total_balls"]
).size().value_counts()

1    275599
Name: count, dtype: int64

In [40]:
balls.loc[
    (balls["matchId"] == 1254073)
    & (balls["inning"] == 1)
    & (balls["over"] == 16)
    & (balls["total_balls"] == 5),
    "batsman_runs",
] = 3
balls.loc[
    (balls["matchId"] == 1254073)
    & (balls["inning"] == 1)
    & (balls["over"] == 16)
    & (balls["total_balls"] == 5),
    "total_runs",
] = 4
balls.loc[
    (balls["matchId"] == 1254073)
    & (balls["inning"] == 1)
    & (balls["over"] == 16)
    & (balls["total_balls"] == 5),
    "current_score",
] = 181
to_drop = balls.loc[
    (balls["matchId"] == 1254073)
    & (balls["inning"] == 1)
    & (balls["over"] == 16)
    & (balls["total_balls"] > 5)
].index

balls = balls.drop(to_drop)

In [41]:
balls.loc[
    (balls["matchId"] == 1178398)
    & (balls["inning"] == 1)
    & (balls["over"] == 17)
    & (balls["total_balls"] == 5),
    "batsman_runs",
] = 2
balls.loc[
    (balls["matchId"] == 1178398)
    & (balls["inning"] == 1)
    & (balls["over"] == 17)
    & (balls["total_balls"] == 5),
    "total_runs",
] = 3
balls.loc[
    (balls["matchId"] == 1178398)
    & (balls["inning"] == 1)
    & (balls["over"] == 17)
    & (balls["total_balls"] == 5),
    "current_score",
] = 111
to_drop = balls.loc[
    (balls["matchId"] == 1178398)
    & (balls["inning"] == 1)
    & (balls["over"] == 17)
    & (balls["total_balls"] > 5)
].index

balls = balls.drop(to_drop)

In [42]:
balls.loc[
    (balls["matchId"] == 729309)
    & (balls["inning"] == 1)
    & (balls["over"] == 18)
    & (balls["total_balls"] == 4),
    "batsman_runs",
] = 6
balls.loc[
    (balls["matchId"] == 729309)
    & (balls["inning"] == 1)
    & (balls["over"] == 18)
    & (balls["total_balls"] == 4),
    "total_runs",
] = 6
balls.loc[
    (balls["matchId"] == 729309)
    & (balls["inning"] == 1)
    & (balls["over"] == 18)
    & (balls["total_balls"] == 4),
    "current_score",
] = 131
to_drop = balls.loc[
    (balls["matchId"] == 729309)
    & (balls["inning"] == 1)
    & (balls["over"] == 18)
    & (balls["total_balls"] > 4)
].index

balls = balls.drop(to_drop)

In [43]:
balls = balls.sort_values(["matchId", "inning", "over", "total_balls"]).reset_index(
    drop=True
)

In [44]:
balls["is_boundary"] = balls["batsman_runs"].isin([4, 6]).astype(int)


def compute_balls_since_boundary(x):
    groups = x.cumsum()
    result = x.groupby(groups).cumcount()
    result[groups == 0] = range((groups == 0).sum())

    return result


balls["balls_since_boundary"] = balls.groupby(["matchId", "inning"])[
    "is_boundary"
].transform(compute_balls_since_boundary)

In [45]:
balls["balls_since_boundary"].unique()

array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
        13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
        26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
        39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
        52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
        65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,
        78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
        91,  92,  93,  94,  95,  96,  97,  98,  99, 100])

In [46]:
len(
    balls[
        (balls["balls_since_boundary"] > 1)
        & (balls["over"] == 0)
        & (balls["total_balls"] == 1)
    ]
)

0

In [47]:
balls["percentage_target_achieved"] = np.where(
    balls["inning"] == 0, 0.0, balls["current_score"] / balls["target"]
)
balls["percentage_target_achieved"] = (
    balls["percentage_target_achieved"].replace([np.inf, -np.inf], 0).fillna(0)
)

In [48]:
balls["percentage_target_achieved"][balls["inning"] == 1].head(2)

129    0.004484
130    0.008969
Name: percentage_target_achieved, dtype: float64

In [49]:
balls = balls.merge(matches[["matchId", "venue"]], on="matchId", how="left")

In [50]:
balls["venue"].head(240)

0                           M Chinnaswamy Stadium
1                           M Chinnaswamy Stadium
2                           M Chinnaswamy Stadium
3                           M Chinnaswamy Stadium
4                           M Chinnaswamy Stadium
                          ...                    
235    Punjab Cricket Association Stadium, Mohali
236    Punjab Cricket Association Stadium, Mohali
237    Punjab Cricket Association Stadium, Mohali
238    Punjab Cricket Association Stadium, Mohali
239    Punjab Cricket Association Stadium, Mohali
Name: venue, Length: 240, dtype: object

In [51]:
balls["balls_bowled"] = balls.groupby(["matchId", "inning"])["legal_ball_1"].cumsum()

balls["overs_bowled"] = balls["balls_bowled"] / 6

balls["current_run_rate"] = np.where(
    balls["balls_bowled"] > 0, balls["current_score"] / balls["overs_bowled"], 0
)

In [52]:
balls["runs_required"] = balls["target"] - balls["current_score"]

balls["overs_remaining"] = balls["balls_remaining"] / 6

balls["required_run_rate"] = balls["runs_required"] / balls["overs_remaining"]

# handle divide by 0
balls["required_run_rate"] = balls["required_run_rate"].replace([np.inf, -np.inf], 0)
balls["required_run_rate"] = balls["required_run_rate"].fillna(0)

balls.loc[balls["inning"] == 0, "required_run_rate"] = 0
balls.loc[balls["runs_required"] <= 0, "required_run_rate"] = 0

In [53]:
balls[["current_run_rate", "required_run_rate"]].describe()

,current_run_rate,required_run_rate
count,275593.000000,275593.000000
mean,7.660384,5.237653
std,2.433127,10.762982
min,0.000000,0.000000
25%,6.396226,0.000000
50%,7.657143,0.000000
75%,8.923077,8.960000
max,66.000000,792.000000


In [54]:
pd.set_option("display.max_columns", None)
balls[balls["current_run_rate"] > 42].head()

,matchId,inning,over,legal_ball,batsman_runs,isWide,isNoBall,Byes,LegByes,Penalty,total_runs,batting_team,bowling_team,batsman,non_striker,bowler,player_dismissed,date,is_legal,legal_ball_1,balls_remaining,over_number,phase,current_score,is_wicket,wickets_fallen,target,last_over_runs,total_balls,is_boundary,balls_since_boundary,percentage_target_achieved,venue,balls_bowled,overs_bowled,current_run_rate,runs_required,overs_remaining,required_run_rate
46911,501222,0,0,1.0,0.0,1,0,0.0,0.0,0.0,1.0,Kolkata Knight Riders,Royal Challengers Bangalore,JH Kallis,BJ Haddin,Z Khan,Not Out,2011-04-22,0,False,119,1,0,8.0,0,0,0.0,0,5,0,4,0.000000,Eden Gardens,1,0.166667,48.0,-8.0,19.833333,0.000000
46912,501222,0,0,1.0,0.0,1,0,0.0,0.0,0.0,1.0,Kolkata Knight Riders,Royal Challengers Bangalore,JH Kallis,BJ Haddin,Z Khan,Not Out,2011-04-22,0,False,119,1,0,9.0,0,0,0.0,0,6,0,5,0.000000,Eden Gardens,1,0.166667,54.0,-9.0,19.833333,0.000000
46913,501222,0,0,1.0,0.0,1,0,0.0,0.0,0.0,1.0,Kolkata Knight Riders,Royal Challengers Bangalore,JH Kallis,BJ Haddin,Z Khan,Not Out,2011-04-22,0,False,119,1,0,10.0,0,0,0.0,0,7,0,6,0.000000,Eden Gardens,1,0.166667,60.0,-10.0,19.833333,0.000000
46914,501222,0,0,1.0,0.0,1,0,0.0,0.0,0.0,1.0,Kolkata Knight Riders,Royal Challengers Bangalore,JH Kallis,BJ Haddin,Z Khan,Not Out,2011-04-22,0,False,119,1,0,11.0,0,0,0.0,0,8,0,7,0.000000,Eden Gardens,1,0.166667,66.0,-11.0,19.833333,0.000000
84877,598034,1,0,1.0,4.0,0,0,4.0,0.0,0.0,4.0,Kolkata Knight Riders,Chennai Super Kings,MS Bisla,G Gambhir,DP Nannes,Not Out,2013-04-28,1,True,119,1,0,10.0,0,0,201.0,0,7,1,0,0.049751,"MA Chidambaram Stadium, Chepauk",1,0.166667,60.0,191.0,19.833333,9.630252


In [55]:
key_cols = ["matchId", "inning", "over", "total_balls"]

multi_batters = (
    balls.groupby(key_cols)["batsman"].nunique().reset_index(name="batter_count")
)

bad_balls_batter = multi_batters[multi_batters["batter_count"] > 1]

print(bad_balls_batter.shape)
bad_balls_batter.head()

(0, 5)


,matchId,inning,over,total_balls,batter_count


In [56]:
multi_bowlers = (
    balls.groupby(key_cols)["bowler"].nunique().reset_index(name="bowler_count")
)

bad_balls_bowler = multi_bowlers[multi_bowlers["bowler_count"] > 1]

print(bad_balls_bowler.shape)
bad_balls_bowler.head()

(0, 5)


,matchId,inning,over,total_balls,bowler_count


In [57]:
balls.index.is_unique

True

In [58]:
mask = (balls["isNoBall"] == 1) & (balls["player_dismissed"] != "Not Out")
balls.loc[mask, "isWide"] = 1
balls.loc[mask, "isNoBall"] = 0
len(balls[(balls["isNoBall"] == 1) & (balls["player_dismissed"] != "Not Out")])

0

In [59]:
balls["over"] = balls["over"] / 20
np.set_printoptions(suppress=True)
balls["over"].unique()

array([0.  , 0.05, 0.1 , 0.15, 0.2 , 0.25, 0.3 , 0.35, 0.4 , 0.45, 0.5 ,
       0.55, 0.6 , 0.65, 0.7 , 0.75, 0.8 , 0.85, 0.9 , 0.95])

In [60]:
season_map = matches.set_index("matchId")["season"]
balls["season"] = balls["matchId"].map(season_map)

In [61]:
# create cyclic features
balls["sin_ball"] = np.sin(2 * np.pi * balls["legal_ball"] / 6)
balls["cos_ball"] = np.cos(2 * np.pi * balls["legal_ball"] / 6)

In [62]:
balls.drop(
    columns=[
        "legal_ball",
        "batting_team",
        "bowling_team",
        "Byes",
        "LegByes",
        "Penalty",
        "over_number",
        "is_legal",
        "legal_ball_1",
        "is_boundary",
        "legal_ball",
        "balls_bowled",
        "overs_bowled",
        "runs_required",
        "overs_remaining",
        "date",
    ],
    inplace=True,
)
balls.head()

,matchId,inning,over,batsman_runs,isWide,isNoBall,total_runs,batsman,non_striker,bowler,player_dismissed,balls_remaining,phase,current_score,is_wicket,wickets_fallen,target,last_over_runs,total_balls,balls_since_boundary,percentage_target_achieved,venue,current_run_rate,required_run_rate,season,sin_ball,cos_ball
0,335982,0,0.0,1.0,0,0,1.0,SC Ganguly,BB McCullum,P Kumar,Not Out,119,0,1.0,0,0,0.0,0,1,0,0.0,M Chinnaswamy Stadium,6.0,0.0,2008,8.660254e-01,0.5
1,335982,0,0.0,0.0,0,0,0.0,BB McCullum,SC Ganguly,P Kumar,Not Out,118,0,1.0,0,0,0.0,0,2,1,0.0,M Chinnaswamy Stadium,3.0,0.0,2008,8.660254e-01,-0.5
2,335982,0,0.0,0.0,1,0,1.0,BB McCullum,SC Ganguly,P Kumar,Not Out,118,0,2.0,0,0,0.0,0,3,2,0.0,M Chinnaswamy Stadium,6.0,0.0,2008,8.660254e-01,-0.5
3,335982,0,0.0,0.0,0,0,0.0,BB McCullum,SC Ganguly,P Kumar,Not Out,117,0,2.0,0,0,0.0,0,4,3,0.0,M Chinnaswamy Stadium,4.0,0.0,2008,1.224647e-16,-1.0
4,335982,0,0.0,0.0,0,0,0.0,BB McCullum,SC Ganguly,P Kumar,Not Out,116,0,2.0,0,0,0.0,0,5,4,0.0,M Chinnaswamy Stadium,3.0,0.0,2008,-8.660254e-01,-0.5


In [63]:
balls["batsman_runs"] = balls["batsman_runs"] / 6
balls["total_runs"] = balls["total_runs"] / 6
balls["balls_remaining"] = balls["balls_remaining"] / 120
balls["wickets_fallen"] = balls["wickets_fallen"] / 10
balls["balls_since_boundary"] = balls["balls_since_boundary"] / 120

In [64]:
balls["current_score"] = balls["current_score"] / 200
balls["target"] = balls["target"] / 200
balls["last_over_runs"] = balls["last_over_runs"] / 200
balls["total_balls"] = balls["total_balls"] / 10

In [65]:
max = 2025 - 2008 + 1
values = np.arange(1, max + 1)
mean = np.mean(values)
std = np.std(values)

balls["season"] = ((balls["season"] - 2007) - mean) / std

In [66]:
balls["current_run_rate"] = balls["current_run_rate"] / 36
balls["required_run_rate"] = balls["required_run_rate"] / 36
balls["required_run_rate"] = balls["required_run_rate"].clip(upper=2)

In [67]:
balls["required_run_rate"].unique().max()

np.float64(2.0)

In [68]:
balls[balls["percentage_target_achieved"] >= 1.05]

,matchId,inning,over,batsman_runs,isWide,isNoBall,total_runs,batsman,non_striker,bowler,player_dismissed,balls_remaining,phase,current_score,is_wicket,wickets_fallen,target,last_over_runs,total_balls,balls_since_boundary,percentage_target_achieved,venue,current_run_rate,required_run_rate,season,sin_ball,cos_ball
147972,1082645,1,0.55,1.0,0,0,1.0,AM Rahane,SPD Smith,GJ Maxwell,Not Out,0.400000,1,0.390,0,0.1,0.37,0.025,0.6,0.0,1.054054,Maharashtra Cricket Association Stadium,0.180556,0.0,0.096374,-2.449294e-16,1.0
220416,1304105,1,0.70,1.0,0,0,1.0,TH David,Tilak Varma,MM Ali,Not Out,0.258333,1,0.515,0,0.5,0.49,0.030,0.5,0.0,1.051020,"Wankhede Stadium, Mumbai",0.192884,0.0,1.060113,-8.660254e-01,0.5


In [69]:
balls.to_csv("data4.csv")